# EEG 特征提取器

这个notebook提供了一个简单的接口，用于从EEG信号中提取各种特征。

## 输入
- `eeg_data`: numpy数组，形状为 `[通道数, 信号点数]`
- `sampling_rate`: 采样率 (Hz)
- `features`: 想要计算的特征名称列表

## 输出
- 特征值字典 `{特征名: 数值}`

## 1. 导入模块

In [15]:
import numpy as np
import pandas as pd
from typing import List, Dict, Optional, Union
from copy import deepcopy

# 导入特征提取模块
from eeg_feature_extraction.feature_extractor import FeatureExtractor
from eeg_feature_extraction.config import Config, FrequencyBands, ChannelGroups
from eeg_feature_extraction.features.base import FeatureRegistry
from selective_feature_extraction import FEATURE_GROUPS as LIB_FEATURE_GROUPS

## 2. EEG特征计算器类

In [16]:
class EEGFeatureCalculator:
    """
    EEG特征计算器 - 简化版接口

    支持从原始EEG信号中提取特征，分为若干特征组（动态对齐主库定义）。
    """

    # 与主库保持一致的特征组定义（包含微状态，名称为 Microstate_0/1/2/3_*）
    FEATURE_GROUPS = {k: v.copy() for k, v in LIB_FEATURE_GROUPS.items()}
    
    def __init__(self, sampling_rate: float = 200.0, use_gpu: bool = False):
        """
        初始化特征计算器

        Args:
            sampling_rate: 采样率 (Hz)，默认200Hz
            use_gpu: 是否使用GPU加速，默认False
        """
        self.sampling_rate = sampling_rate
        self.use_gpu = use_gpu
        self._extractor = None
        self._config = None
        
    def _build_extractor(self, n_channels: int):
        """根据通道数构建提取器"""
        self._config = Config(
            sampling_rate=self.sampling_rate,
            n_channels=n_channels,
            use_gpu=self.use_gpu
        )
        # 生成通道名称
        self._config.channel_names = [f"CH{i+1}" for i in range(n_channels)]
        self._extractor = FeatureExtractor(self._config, n_jobs=1)
        
    def _get_feature_groups_for_features(self, features: List[str]) -> List[str]:
        """根据特征名称获取需要计算的特征组"""
        groups = set()
        for feature in features:
            for group_name, group_features in self.FEATURE_GROUPS.items():
                if feature in group_features:
                    groups.add(group_name)
                    break
        return list(groups)
    
    def calculate(self, 
                  eeg_data: np.ndarray, 
                  features: Optional[Union[List[str], str]] = None,
                  sampling_rate: Optional[float] = None) -> Dict[str, float]:
        """
        计算EEG特征

        Args:
            eeg_data: EEG数据，形状为 [通道数, 信号点数]
            features: 想要计算的特征，可以是:
                - None: 计算所有特征
                - str: 单个特征名或特征组名 (如 'alpha_power' 或 'frequency_domain')
                - List[str]: 特征名或特征组名的列表
            sampling_rate: 可选的采样率覆盖，如果提供则更新内部采样率
                
        Returns:
            Dict[str, float]: 特征名到值的字典
            
        Example:
            >>> calc = EEGFeatureCalculator(sampling_rate=256)
            >>> data = np.random.randn(62, 512)  # 62通道，2秒数据
            >>> result = calc.calculate(data, features=['alpha_power', 'theta_power'])
            >>> print(result)
            {'alpha_power': 0.123, 'theta_power': 0.456}
        """
        # 验证输入
        if not isinstance(eeg_data, np.ndarray):
            raise TypeError("eeg_data must be a numpy array")
        if eeg_data.ndim != 2:
            raise ValueError(f"eeg_data must be 2D [n_channels, n_timepoints], got shape {eeg_data.shape}")
            
        n_channels, n_timepoints = eeg_data.shape
        
        # 更新采样率
        if sampling_rate is not None:
            self.sampling_rate = sampling_rate
            
        # 构建或重建提取器
        if self._extractor is None or self._config.n_channels != n_channels:
            self._build_extractor(n_channels)
        elif self._config.sampling_rate != self.sampling_rate:
            self._build_extractor(n_channels)
            
        # 处理特征参数
        if features is None:
            # 计算所有特征
            feature_groups = None
            requested_features = None
        else:
            if isinstance(features, str):
                features = [features]
                
            # 区分特征组和单个特征
            feature_groups = []
            requested_features = []
            
            for f in features:
                if f in self.FEATURE_GROUPS:
                    # 是特征组名
                    feature_groups.append(f)
                else:
                    # 是单个特征名
                    requested_features.append(f)
                    
            # 为单个特征找到对应的组
            if requested_features:
                additional_groups = self._get_feature_groups_for_features(requested_features)
                feature_groups.extend(additional_groups)
                feature_groups = list(set(feature_groups))
                
            if not feature_groups:
                raise ValueError(f"No valid features found. Check feature names.")
                
        # 提取特征
        all_features = self._extractor.extract_features(eeg_data, feature_groups)
        
        # 如果指定了具体特征，只返回这些特征
        if requested_features:
            result = {}
            # 首先添加请求的单个特征
            for f in requested_features:
                if f in all_features:
                    result[f] = all_features[f]
                else:
                    raise ValueError(f"Feature '{f}' not found. Available features: {list(all_features.keys())}")
            # 添加请求的整个特征组
            for group in features:
                if group in self.FEATURE_GROUPS:
                    for f in self.FEATURE_GROUPS[group]:
                        if f in all_features:
                            result[f] = all_features[f]
            return result
        else:
            return all_features
    
    @classmethod
    def list_all_features(cls) -> pd.DataFrame:
        """
        列出所有可用的特征
        
        Returns:
            DataFrame 包含特征名和所属组
        """
        data = []
        for group, features in cls.FEATURE_GROUPS.items():
            for feature in features:
                data.append({'feature_name': feature, 'group': group})
        return pd.DataFrame(data)
    
    @classmethod
    def list_feature_groups(cls) -> List[str]:
        """
        列出所有特征组名称
        
        Returns:
            特征组名称列表
        """
        return list(cls.FEATURE_GROUPS.keys())
    
    @classmethod  
    def get_features_in_group(cls, group_name: str) -> List[str]:
        """
        获取特定组中的所有特征
        
        Args:
            group_name: 特征组名称
            
        Returns:
            该组中的特征名列表
        """
        if group_name not in cls.FEATURE_GROUPS:
            raise ValueError(f"Unknown group: {group_name}. Available groups: {list(cls.FEATURE_GROUPS.keys())}")
        return cls.FEATURE_GROUPS[group_name].copy()

## 3. 查看所有可用特征

In [17]:
# 查看所有特征组
print("可用的特征组:")
print(EEGFeatureCalculator.list_feature_groups())

可用的特征组:
['time_domain', 'frequency_domain', 'complexity', 'connectivity', 'network', 'composite']


In [18]:
# 查看所有特征的详细信息
all_features = EEGFeatureCalculator.list_all_features()
print(f"\n共 {len(all_features)} 个特征:")
all_features


共 44 个特征:


,feature_name,group
0,mean_abs_amplitude,time_domain
1,mean_channel_std,time_domain
2,mean_peak_to_peak,time_domain
3,mean_rms,time_domain
4,mean_zero_crossing_rate,time_domain
5,hjorth_activity,time_domain
6,hjorth_mobility,time_domain
7,hjorth_complexity,time_domain
8,delta_power,frequency_domain
9,theta_power,frequency_domain


In [19]:
# 查看特定组的特征
print("频域特征:")
print(EEGFeatureCalculator.get_features_in_group('frequency_domain'))

频域特征:
['delta_power', 'theta_power', 'alpha_power', 'beta_power', 'gamma_power', 'delta_relative_power', 'theta_relative_power', 'alpha_relative_power', 'beta_relative_power', 'gamma_relative_power', 'peak_frequency', 'spectral_entropy', 'spectral_centroid', 'individual_alpha_frequency', 'theta_beta_ratio', 'delta_theta_ratio', 'low_high_power_ratio', 'aperiodic_exponent', 'mean_total_power']


## 4. 使用示例

### 4.1 生成模拟EEG数据

In [20]:
# 生成模拟EEG数据
np.random.seed(42)

n_channels = 62       # 通道数
sampling_rate = 200   # 采样率 (Hz)
duration = 2.0        # 持续时间 (秒)
n_timepoints = int(sampling_rate * duration)  # 信号点数

# 创建包含alpha波(10Hz)的模拟信号
t = np.linspace(0, duration, n_timepoints)
eeg_data = np.zeros((n_channels, n_timepoints))

for ch in range(n_channels):
    # 添加alpha波 (8-13 Hz)
    alpha = 0.5 * np.sin(2 * np.pi * 10 * t + np.random.rand() * 2 * np.pi)
    # 添加theta波 (4-8 Hz)
    theta = 0.3 * np.sin(2 * np.pi * 6 * t + np.random.rand() * 2 * np.pi)
    # 添加噪声
    noise = 0.1 * np.random.randn(n_timepoints)
    eeg_data[ch] = alpha + theta + noise

print(f"EEG数据形状: {eeg_data.shape}")
print(f"采样率: {sampling_rate} Hz")
print(f"时长: {duration} 秒")

EEG数据形状: (62, 400)
采样率: 200 Hz
时长: 2.0 秒


### 4.2 计算特定特征

In [21]:
# 创建特征计算器
calc = EEGFeatureCalculator(sampling_rate=sampling_rate)

# 计算单个特征
result = calc.calculate(eeg_data, features='alpha_power')
print("单个特征:")
print(result)

CuPy 不可用，使用 CPU 计算
CuPy 不可用，使用 CPU 计算
CuPy 不可用，使用 CPU 计算
单个特征:
{'alpha_power': 0.1254442951080106}


In [22]:
# 计算多个特征
result = calc.calculate(eeg_data, features=['alpha_power', 'theta_power', 'theta_beta_ratio'])
print("多个特征:")
for name, value in result.items():
    print(f"  {name}: {value:.6f}")

多个特征:
  alpha_power: 0.125444
  theta_power: 0.045902
  theta_beta_ratio: 28.273256


### 4.3 计算整个特征组

In [23]:
# 计算整个时域特征组
result = calc.calculate(eeg_data, features='time_domain')
print("时域特征:")
for name, value in result.items():
    print(f"  {name}: {value:.6f}")

时域特征:
  mean_abs_amplitude: 0.355179
  mean_channel_std: 0.424710
  mean_peak_to_peak: 1.902282
  mean_rms: 0.424740
  mean_zero_crossing_rate: 24.024194
  hjorth_activity: 0.180405
  hjorth_mobility: 0.433725
  hjorth_complexity: 3.092665


In [24]:
# 计算频域特征组
result = calc.calculate(eeg_data, features='frequency_domain')
print("频域特征:")
for name, value in result.items():
    print(f"  {name}: {value:.6f}")

频域特征:
  delta_power: 0.000312
  theta_power: 0.045902
  alpha_power: 0.125444
  beta_power: 0.001624
  gamma_power: 0.006990
  delta_relative_power: 0.001732
  theta_relative_power: 0.254411
  alpha_relative_power: 0.695117
  beta_relative_power: 0.009004
  gamma_relative_power: 0.038755
  peak_frequency: 10.156250
  spectral_entropy: 2.509857
  spectral_centroid: 11.238491
  individual_alpha_frequency: 10.156250
  theta_beta_ratio: 28.273256
  delta_theta_ratio: 0.006790
  low_high_power_ratio: 18.332764
  aperiodic_exponent: 1.242037
  mean_total_power: 0.180448


### 4.4 计算所有特征

In [25]:
# 计算所有特征
all_result = calc.calculate(eeg_data)  # features=None 表示计算所有
print(f"共计算了 {len(all_result)} 个特征:")

# 转为DataFrame方便查看
df = pd.DataFrame([all_result]).T
df.columns = ['value']
df

/mnt/dataset4/cx/code/EEG_LLM_text/eeg_feature_extraction/features/microstate.py:579: UserWarning: 未提供预计算的微状态模板，将从当前 segment 生成模板。建议在 subject 级别预计算模板以获得更稳定的结果。
  warnings.warn(


共计算了 64 个特征:


,value
mean_abs_amplitude,0.355179
mean_channel_std,0.424710
mean_peak_to_peak,1.902282
mean_rms,0.424740
mean_zero_crossing_rate,24.024194
...,...
Microstate_3_meandurs,0.014714
Microstate_3_occurrence,17.500000
Microstate_3_timecov,0.515000
Microstate_3_mean_corr,0.821789


### 4.5 混合使用特征名和特征组名

In [26]:
# 可以混合使用特征名和特征组名
result = calc.calculate(
    eeg_data, 
    features=['time_domain', 'alpha_power', 'spectral_entropy']  # 一个组 + 两个单独特征
)
print(f"计算了 {len(result)} 个特征:")
for name, value in result.items():
    print(f"  {name}: {value:.6f}")

计算了 10 个特征:
  alpha_power: 0.125444
  spectral_entropy: 2.509857
  mean_abs_amplitude: 0.355179
  mean_channel_std: 0.424710
  mean_peak_to_peak: 1.902282
  mean_rms: 0.424740
  mean_zero_crossing_rate: 24.024194
  hjorth_activity: 0.180405
  hjorth_mobility: 0.433725
  hjorth_complexity: 3.092665


## 5. 便捷函数

In [27]:
def extract_eeg_features(
    eeg_data: np.ndarray,
    sampling_rate: float,
    features: Optional[Union[List[str], str]] = None
) -> Dict[str, float]:
    """
    便捷函数：从EEG数据提取特征
    
    Args:
        eeg_data: EEG数据，形状为 [通道数, 信号点数]
        sampling_rate: 采样率 (Hz)
        features: 想要计算的特征（可选）:
            - None: 计算所有64个特征
            - str: 单个特征名或特征组名
            - List[str]: 多个特征名或特征组名
            
    Returns:
        Dict[str, float]: 特征名到值的字典
        
    可用特征组:
        - 'time_domain': 时域特征 (8个)
        - 'frequency_domain': 频域特征 (19个) 
        - 'complexity': 复杂度特征 (4个)
        - 'connectivity': 连接性特征 (6个)
        - 'network': 网络特征 (4个)
        - 'composite': 综合特征 (3个)
        - 'microstate': 微状态特征 (20个)
        
    Example:
        >>> data = np.random.randn(62, 512)
        >>> result = extract_eeg_features(data, 256, ['alpha_power', 'theta_power'])
        >>> print(result)
    """
    calc = EEGFeatureCalculator(sampling_rate=sampling_rate)
    return calc.calculate(eeg_data, features=features)

In [28]:
# 使用便捷函数
result = extract_eeg_features(
    eeg_data=eeg_data,
    sampling_rate=200,
    features=['alpha_power', 'theta_power', 'hjorth_complexity']
)
print(result)

CuPy 不可用，使用 CPU 计算
CuPy 不可用，使用 CPU 计算
CuPy 不可用，使用 CPU 计算
{'alpha_power': 0.1254442951080106, 'theta_power': 0.04590168695137769, 'hjorth_complexity': 3.092665120916082}


## 6. 特征说明

### 时域特征 (time_domain)
| 特征名 | 说明 |
|--------|------|
| mean_abs_amplitude | 平均绝对幅度 |
| mean_channel_std | 通道平均标准差 |
| mean_peak_to_peak | 平均峰峰值 |
| mean_rms | 平均均方根 |
| mean_zero_crossing_rate | 平均过零率 |
| hjorth_activity | Hjorth活动性参数（信号方差）|
| hjorth_mobility | Hjorth移动性参数（频率内容）|
| hjorth_complexity | Hjorth复杂度参数（信号形状）|

### 频域特征 (frequency_domain)
| 特征名 | 说明 |
|--------|------|
| delta_power | Delta波段(0.5-4Hz)功率 |
| theta_power | Theta波段(4-8Hz)功率 |
| alpha_power | Alpha波段(8-13Hz)功率 |
| beta_power | Beta波段(13-30Hz)功率 |
| gamma_power | Gamma波段(30-100Hz)功率 |
| *_relative_power | 各波段相对功率 |
| peak_frequency | 峰值频率 |
| spectral_entropy | 频谱熵 |
| spectral_centroid | 频谱重心 |
| individual_alpha_frequency | 个体alpha频率(IAF) |
| theta_beta_ratio | θ/β比值 |
| delta_theta_ratio | δ/θ比值 |
| low_high_power_ratio | 低频/高频功率比 |
| aperiodic_exponent | 非周期性指数(1/f斜率) |
| mean_total_power | 平均总功率 |

### 复杂度特征 (complexity)
| 特征名 | 说明 |
|--------|------|
| wavelet_energy_entropy | 小波能量熵 |
| sample_entropy | 样本熵 |
| approx_entropy | 近似熵 |
| hurst_exponent | Hurst指数 |

### 连接性特征 (connectivity)
| 特征名 | 说明 |
|--------|------|
| mean_interchannel_correlation | 通道间平均相关性 |
| mean_alpha_coherence | Alpha波段平均相干性 |
| interhemispheric_alpha_coherence | 半球间Alpha相干性 |
| alpha_beta_band_power_correlation | Alpha-Beta功率相关 |
| hemispheric_alpha_asymmetry | 半球Alpha不对称性 |
| frontal_occipital_alpha_ratio | 额枕Alpha比值 |

### 网络特征 (network)
| 特征名 | 说明 |
|--------|------|
| network_clustering_coefficient | 网络聚类系数 |
| network_characteristic_path_length | 网络特征路径长度 |
| network_global_efficiency | 网络全局效率 |
| network_small_world_index | 小世界指数 |

### 综合特征 (composite)
| 特征名 | 说明 |
|--------|------|
| cognitive_load_estimate | 认知负荷估计(0-1) |
| alertness_estimate | 警觉度估计(0-1) |
| relaxation_index | 放松指数(0-1) |

### 微状态特征 (microstate)
微状态命名与主库保持一致，使用数字索引 0/1/2/3（对应 A/B/C/D），字段名示例：`Microstate_0_meandurs`。
对于每个微状态(0-3):
| 特征名 | 说明 |
|--------|------|
| Microstate_X_meandurs | 平均持续时间(秒) |
| Microstate_X_occurrence | 发生频率(次/秒) |
| Microstate_X_timecov | 时间覆盖率 |
| Microstate_X_mean_corr | 与模板的平均相关性 |
| Microstate_X_gev | 全局解释方差 |